# Train the "Hey Robin" wake word

A microWakeWord model for the Robin Egg (XIAO ESP32-S3 + ReSpeaker XVF3800, formatBCE ESPHome firmware). It's built on the upstream `basic_training_notebook.ipynb`, with these changes:

- **Several spellings** of "Hey Robin" go to Piper, so the synthetic voices vary more.
- **Household confusables** are trained as negatives: "Hey Ryan", "Robin" on its own (the TV), "Hey Siri" (the HomePod), "robbing", "Robert" and "hey Rob".
- **Real recordings** of Tanner and Ryan are used if they're in Drive. Long Voice Memos are cut into single takes automatically.
- **Results are saved to Google Drive**, so a disconnect doesn't lose work and a second run is faster.
- **A real-voice check** at the end reports how often each person triggers it, and how often their "don't wake" takes trigger it.
- **The ESPHome manifest** (`hey_robin.json`) is written next to the `.tflite`.

## How to run
1. **Runtime → Change runtime type → T4 GPU.**
2. Optional: put recordings in Drive at `MyDrive/robin_wakeword/recordings/`. Name each file with the speaker and `positive` or `negative`, e.g. `tanner_positive.m4a`, `ryan_negative.m4a`. Any audio format works.
3. **Runtime → Run all.** The install cell restarts the runtime once. When it does, choose **Run all** again.
4. Allow Drive access when asked.

The first full run takes about 2–3 hours, mostly downloads, sample generation and training. Later runs reuse the cached samples from Drive.

> The upstream negative and augmentation datasets are for **non-commercial personal use only**. That's fine for our own egg. A product we sell would need its own data.

In [ ]:
# ---- Settings: change these between rounds ----
MODEL_VERSION = "v1"          # bump each round; outputs go to Drive/robin_wakeword/outputs/<version>/

# Spellings fed to Piper for the wake word. Phonetic spellings give more variety.
POSITIVE_PHRASES = ["hey robin", "hey robbin", "hey rob in", "hay robin"]
SAMPLES_PER_POSITIVE_PHRASE = 2500

# Things said in this house that must NOT wake the egg.
NEGATIVE_PHRASES = [
    "hey ryan", "hey rion", "robin", "a robin", "the robin", "robbing",
    "hey robert", "hey rob", "hey ruben", "hey bobbin", "hey rowan",
    "hey siri", "siri", "hey", "hey there", "rob and", "hey roman",
]
SAMPLES_PER_NEGATIVE_PHRASE = 800

TRAINING_STEPS = 15000
NEGATIVE_CLASS_WEIGHT = 20     # raise to cut false wakes, at the cost of more misses
PROBABILITY_CUTOFF = 0.97      # written to the manifest; tune from the report at the end

DRIVE_DIR = "/content/drive/MyDrive/robin_wakeword"
WORK_DIR = "/content/work"     # fast local disk; anything worth keeping is copied to Drive

In [ ]:
# Install. On a fresh runtime this installs everything and restarts the runtime.
# After the restart, choose Runtime → Run all again; this cell will then skip.
import importlib.util, os, subprocess, sys

def _sh(cmd):
    """Runs a shell command and shows its output in the cell as it happens."""
    print("$", cmd, flush=True)
    p = subprocess.Popen(cmd, shell=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
    for chunk in iter(lambda: p.stdout.read(256), ""):
        sys.stdout.write(chunk); sys.stdout.flush()
    if p.wait():
        raise RuntimeError(f"This command failed with exit status {p.returncode}. The error is in the output above: {cmd}")

needs = {"microwakeword": importlib.util.find_spec("microwakeword") is None,
         "piper": importlib.util.find_spec("piper") is None}
if any(needs.values()):
    if needs["microwakeword"]:
        _sh("pip -q install 'git+https://github.com/whatsnowplaying/audio-metadata@d4ebb238e6a401bb1a5aaaac60c9e2b3cb30929f'")
        if not os.path.exists("/content/microWakeWord"):
            _sh("git clone -q https://github.com/kahrendt/microWakeWord /content/microWakeWord")
        _sh("pip -q install -e /content/microWakeWord")
    # piper-tts brings its own espeak phonemizer and has builds for every recent Python,
    # unlike piper-phonemize, which stops at 3.12.
    _sh("pip -q install 'piper-tts>=1.3.0,<2'")
    print("Installed. Restarting the runtime now: choose Runtime → Run all again.")
    os.kill(os.getpid(), 9)
else:
    print("microWakeWord and Piper already installed.")


In [ ]:
import os, glob, shutil, subprocess, sys, json, re
from pathlib import Path
from google.colab import drive

drive.mount("/content/drive")
for d in [DRIVE_DIR, f"{DRIVE_DIR}/recordings", f"{DRIVE_DIR}/cache", f"{DRIVE_DIR}/outputs", WORK_DIR]:
    os.makedirs(d, exist_ok=True)
os.chdir(WORK_DIR)

def sh(cmd):
    """Runs a shell command and shows its output in the cell as it happens."""
    print("$", cmd, flush=True)
    p = subprocess.Popen(cmd, shell=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
    for chunk in iter(lambda: p.stdout.read(256), ""):
        sys.stdout.write(chunk); sys.stdout.flush()
    if p.wait():
        raise RuntimeError(f"This command failed with exit status {p.returncode}. The error is in the output above: {cmd}")

def slug(text):
    return re.sub(r"[^a-z0-9]+", "_", text.lower()).strip("_")

def restore_cache(name):
    tar = f"{DRIVE_DIR}/cache/{name}.tar"
    if not os.path.exists(name) and os.path.exists(tar):
        sh(f"tar -xf '{tar}' -C '{WORK_DIR}'")

def save_cache(name):
    sh(f"tar -cf '{DRIVE_DIR}/cache/{name}.tar' -C '{WORK_DIR}' '{name}'")

import torch
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NONE. Switch the runtime to a GPU.")

## 1. Synthetic samples (Piper)

In [ ]:
import sys
GENERATOR = "piper-sample-generator/models/en_US-libritts_r-medium.pt"
if not os.path.exists("piper-sample-generator"):
    sh("git clone -q -b v3.1.0 https://github.com/rhasspy/piper-sample-generator")
    sh(f"wget -q -O {GENERATOR} "
       "'https://github.com/rhasspy/piper-sample-generator/releases/download/v2.0.0/en_US-libritts_r-medium.pt'")
if "piper-sample-generator/" not in sys.path:
    sys.path.append("piper-sample-generator/")

def generate(phrases, root, n):
    for phrase in phrases:
        out = f"{root}/{slug(phrase)}"
        have = len(glob.glob(f"{out}/*.wav"))
        if have >= n:
            print(f"{phrase!r}: {have} samples cached"); continue
        sh(f"python3 piper-sample-generator/generate_samples.py '{phrase}' "
           f"--model {GENERATOR} --max-samples {n} --batch-size 100 --output-dir '{out}'")

restore_cache("generated")
generate(POSITIVE_PHRASES, "generated/positive", SAMPLES_PER_POSITIVE_PHRASE)
generate(NEGATIVE_PHRASES, "generated/negative", SAMPLES_PER_NEGATIVE_PHRASE)
save_cache("generated")

In [ ]:
# Listen to one sample per phrase. If a spelling sounds wrong, drop it from POSITIVE_PHRASES.
from IPython.display import Audio, display
for phrase in POSITIVE_PHRASES + NEGATIVE_PHRASES[:4]:
    files = sorted(glob.glob(f"generated/*/{slug(phrase)}/*.wav"))
    if files:
        print(phrase); display(Audio(files[0]))

## 2. Real recordings of Tanner and Ryan (optional, but strongly recommended)

In [ ]:
# Converts every file in Drive/robin_wakeword/recordings/ to 16 kHz mono and cuts it into
# single takes at the silences. A filename containing "negative" means don't-wake takes;
# anything else counts as "Hey Robin". The cut clips are saved to Drive so you can check them.
import numpy as np, librosa, soundfile as sf

REAL_DIR = "real"
shutil.rmtree(REAL_DIR, ignore_errors=True)
counts = {}
for src in sorted(glob.glob(f"{DRIVE_DIR}/recordings/*")):
    if os.path.isdir(src):
        continue
    kind = "negative" if "negative" in os.path.basename(src).lower() else "positive"
    name = slug(Path(src).stem)
    tmp = f"/tmp/{name}.wav"
    sh(f"ffmpeg -loglevel error -y -i '{src}' -ac 1 -ar 16000 '{tmp}'")
    audio, sr = librosa.load(tmp, sr=16000)
    # Find speech regions, merge ones less than 0.35 s apart, keep 0.3–3 s takes.
    regions = librosa.effects.split(audio, top_db=35, frame_length=1024, hop_length=160)
    merged = []
    for s, e in regions:
        if merged and s - merged[-1][1] < int(0.35 * sr):
            merged[-1][1] = e
        else:
            merged.append([s, e])
    out_dir = f"{REAL_DIR}/{kind}"
    os.makedirs(out_dir, exist_ok=True)
    n = 0
    for s, e in merged:
        if not (0.3 * sr <= e - s <= 3.0 * sr):
            continue
        clip = audio[max(0, s - int(0.2 * sr)): e + int(0.2 * sr)]
        sf.write(f"{out_dir}/{name}_{n:03d}.wav", clip, sr, subtype="PCM_16")
        n += 1
    counts[os.path.basename(src)] = (kind, n)

for f, (kind, n) in counts.items():
    print(f"{f}: {n} {kind} takes")
if not counts:
    print("No recordings found, so training is synthetic only.")
else:
    shutil.rmtree(f"{DRIVE_DIR}/recordings_cut", ignore_errors=True)
    shutil.copytree(REAL_DIR, f"{DRIVE_DIR}/recordings_cut")

## 3. Background noise and room echo for augmentation

In [ ]:
# Same sources as the upstream notebook (MIT room impulses, one AudioSet shard, FMA music).
# Cached to Drive as a tar after the first run.
import scipy.io.wavfile
from huggingface_hub import snapshot_download
from tqdm import tqdm

def to_wavs(files, out_dir):
    """Converts audio files to 16 kHz mono WAVs in out_dir. Skips files that can't be read."""
    partial = out_dir + ".partial"   # renamed at the end, so a failed run never looks finished
    shutil.rmtree(partial, ignore_errors=True)
    shutil.rmtree(out_dir, ignore_errors=True)
    os.makedirs(partial)
    skipped = 0
    for src in tqdm(sorted(str(f) for f in files), desc=os.path.basename(out_dir)):
        try:
            audio, _ = librosa.load(src, sr=16000, mono=True)
        except Exception:
            skipped += 1
            continue
        scipy.io.wavfile.write(os.path.join(partial, Path(src).stem + ".wav"), 16000, (np.clip(audio, -1, 1) * 32767).astype(np.int16))
    os.rename(partial, out_dir)
    print(f"{out_dir}: {len(os.listdir(out_dir))} files, {skipped} unreadable files skipped")

def ready(out_dir):
    return bool(glob.glob(f"{out_dir}/*.wav"))

restore_cache("augment")
os.makedirs("augment", exist_ok=True)
if not ready("augment/mit_rirs"):
    rirs = snapshot_download("davidscripka/MIT_environmental_impulse_responses", repo_type="dataset", local_dir="/tmp/mit_rirs")
    to_wavs(Path(rirs).glob("**/*.wav"), "augment/mit_rirs")
if not ready("augment/audioset_16k"):
    # One shard of AudioSet's balanced training set: about 700 MB of 10-second clips, stored as a parquet table.
    import pyarrow.parquet as pq
    sh("mkdir -p /tmp/audioset/audio && wget -q -O /tmp/audioset/09.parquet https://huggingface.co/datasets/agkphysics/AudioSet/resolve/main/data/bal_train/09.parquet")
    for batch in pq.ParquetFile("/tmp/audioset/09.parquet").iter_batches(batch_size=64, columns=["video_id", "audio"]):
        for row in batch.to_pylist():
            if row["audio"] and row["audio"].get("bytes"):
                with open(f"/tmp/audioset/audio/{row['video_id']}.flac", "wb") as f:
                    f.write(row["audio"]["bytes"])
    to_wavs(Path("/tmp/audioset/audio").glob("*.flac"), "augment/audioset_16k")
if not ready("augment/fma_16k"):
    sh("mkdir -p /tmp/fma && wget -q -O /tmp/fma/fma_xs.zip https://huggingface.co/datasets/mchl914/fma_xsmall/resolve/main/fma_xs.zip && cd /tmp/fma && unzip -q -o fma_xs.zip")
    to_wavs(Path("/tmp/fma").glob("**/*.mp3"), "augment/fma_16k")
for d in ["augment/mit_rirs", "augment/audioset_16k", "augment/fma_16k"]:
    assert ready(d), f"{d} has no audio files. The download above failed or its layout changed."
save_cache("augment")


## 4. Spectrogram features

In [ ]:
from microwakeword.audio.augmentation import Augmentation
from microwakeword.audio.clips import Clips
from microwakeword.audio.spectrograms import SpectrogramGeneration
from mmap_ninja.ragged import RaggedMmap

augmenter = Augmentation(
    augmentation_duration_s=3.2,
    augmentation_probabilities={
        "SevenBandParametricEQ": 0.1, "TanhDistortion": 0.1, "PitchShift": 0.1,
        "BandStopFilter": 0.1, "AddColorNoise": 0.1, "AddBackgroundNoise": 0.75,
        "Gain": 1.0, "RIR": 0.5,
    },
    impulse_paths=["augment/mit_rirs"],
    background_paths=["augment/fma_16k", "augment/audioset_16k"],
    background_min_snr_db=-5, background_max_snr_db=10,
    min_jitter_s=0.195, max_jitter_s=0.205,
)

def build_features(input_dir, out_dir, train_repeat=2, split_count=0.1):
    """Augment every clip under input_dir into training/validation/testing mmaps under out_dir."""
    if not glob.glob(f"{input_dir}/**/*.wav", recursive=True):
        return False
    shutil.rmtree(out_dir, ignore_errors=True)
    clips = Clips(input_directory=input_dir, file_pattern="**/*.wav", max_clip_duration_s=None,
                  remove_silence=False, random_split_seed=10, split_count=split_count)
    for split, split_name, repeat, slide in [("training", "train", train_repeat, 10),
                                             ("validation", "validation", 1, 10),
                                             ("testing", "test", 1, 1)]:
        gen = SpectrogramGeneration(clips=clips, augmenter=augmenter, slide_frames=slide, step_ms=10)
        os.makedirs(f"{out_dir}/{split}", exist_ok=True)   # RaggedMmap only creates the last folder
        RaggedMmap.from_generator(out_dir=f"{out_dir}/{split}/wakeword_mmap",
                                  sample_generator=gen.spectrogram_generator(split=split_name, repeat=repeat),
                                  batch_size=100, verbose=True)
    return True

FEATURES = {}
FEATURES["synthetic_positive"] = build_features("generated/positive", "features/synthetic_positive")
FEATURES["synthetic_negative"] = build_features("generated/negative", "features/synthetic_negative")
# Real takes are few, so they get many augmented repeats and a bigger held-out share.
FEATURES["real_positive"] = build_features(f"{REAL_DIR}/positive", "features/real_positive", train_repeat=20, split_count=0.2)
FEATURES["real_negative"] = build_features(f"{REAL_DIR}/negative", "features/real_negative", train_repeat=20, split_count=0.2)
print(FEATURES)

In [ ]:
# Pre-made negative features from the microWakeWord authors (speech, dinner party, no speech).
# About 6 GB. Not cached to Drive: downloading again is quicker than copying that much to and from Drive.
if not os.path.exists("negative_datasets/speech"):
    os.makedirs("negative_datasets", exist_ok=True)
    for fname in ["dinner_party.zip", "dinner_party_eval.zip", "no_speech.zip", "speech.zip"]:
        sh(f"wget -q -O negative_datasets/{fname} https://huggingface.co/datasets/kahrendt/microwakeword/resolve/main/{fname}")
        sh(f"unzip -q -o negative_datasets/{fname} -d negative_datasets && rm negative_datasets/{fname}")


## 5. Train

In [ ]:
import yaml

def feat(path, sampling, penalty, truth, trunc):
    return {"features_dir": path, "sampling_weight": sampling, "penalty_weight": penalty,
            "truth": truth, "truncation_strategy": trunc, "type": "mmap"}

features = [feat("features/synthetic_positive", 2.0, 1.0, True, "truncate_start")]
if FEATURES.get("real_positive"):
    features.append(feat("features/real_positive", 1.0, 1.0, True, "truncate_start"))
if FEATURES.get("synthetic_negative"):
    # Confusable phrases are penalised harder: these are the false wakes we expect at home.
    features.append(feat("features/synthetic_negative", 4.0, 2.0, False, "truncate_start"))
if FEATURES.get("real_negative"):
    features.append(feat("features/real_negative", 1.0, 3.0, False, "truncate_start"))
features += [
    feat("negative_datasets/speech", 10.0, 1.0, False, "random"),
    feat("negative_datasets/dinner_party", 10.0, 1.0, False, "random"),
    feat("negative_datasets/no_speech", 5.0, 1.0, False, "random"),
    feat("negative_datasets/dinner_party_eval", 0.0, 1.0, False, "split"),
]

TRAIN_DIR = f"trained_models/hey_robin_{MODEL_VERSION}"
config = {
    "window_step_ms": 10,
    "train_dir": TRAIN_DIR,
    "features": features,
    "training_steps": [TRAINING_STEPS],
    "positive_class_weight": [1],
    "negative_class_weight": [NEGATIVE_CLASS_WEIGHT],
    "learning_rates": [0.001],
    "batch_size": 128,
    "time_mask_max_size": [0], "time_mask_count": [0],
    "freq_mask_max_size": [0], "freq_mask_count": [0],
    "eval_step_interval": 500,
    "clip_duration_ms": 1500,
    # Pick weights with under 0.5 false wakes/hour on ambient audio first, then best recall.
    "target_minimization": 0.5,
    "minimization_metric": "ambient_false_positives_per_hour",
    "maximization_metric": "average_viable_recall",
}
with open("training_parameters.yaml", "w") as f:
    yaml.dump(config, f)
print(yaml.dump(config))

In [ ]:
# Colab doesn't print per-batch progress, so this can look stuck for minutes at a time.
# Checkpoints are copied to Drive at the end. If the runtime drops, rerun from the top.
# The cached samples and features make the rerun much faster.
restore_cache(f"trained_models")
sh("python -m microwakeword.model_train_eval --training_config=training_parameters.yaml "
   "--train 1 --restore_checkpoint 1 "
   "--test_tf_nonstreaming 0 --test_tflite_nonstreaming 0 --test_tflite_nonstreaming_quantized 0 "
   "--test_tflite_streaming 0 --test_tflite_streaming_quantized 1 "
   "--use_weights best_weights "
   "mixednet --pointwise_filters '64,64,64,64' --repeat_in_block '1, 1, 1, 1' "
   "--mixconv_kernel_sizes '[5], [7,11], [9,15], [23]' --residual_connection '0,0,0,0' "
   "--first_conv_filters 32 --first_conv_kernel_size 5 --stride 3")
save_cache("trained_models")

## 6. Check against Tanner's and Ryan's real voices

In [ ]:
# Runs the streaming model on each real take (padded with silence) and reports, per file,
# the share of "Hey Robin" takes that wake it and the share of don't-wake takes that falsely do.
# Uses the same 5-window sliding average the firmware uses.
from microwakeword.inference import Model

TFLITE = f"{TRAIN_DIR}/tflite_stream_state_internal_quant/stream_state_internal_quant.tflite"
model = Model(TFLITE)
cutoffs = [0.80, 0.90, 0.95, 0.97, 0.99]

def peak(path):
    audio, _ = librosa.load(path, sr=16000)
    audio = np.concatenate([np.zeros(16000), audio, np.zeros(8000)])
    probs = np.array(model.predict_clip((audio * 32767).astype(np.int16), step_ms=10))
    if len(probs) < 5:
        return float(probs.max()) if len(probs) else 0.0
    return float(np.convolve(probs, np.ones(5) / 5, mode="valid").max())

for kind in ["positive", "negative"]:
    by_source = {}
    for f in sorted(glob.glob(f"{REAL_DIR}/{kind}/*.wav")):
        source = Path(f).stem.rsplit("_", 1)[0]
        by_source.setdefault(source, []).append(peak(f))
    for source, peaks in by_source.items():
        peaks = np.array(peaks)
        rates = "  ".join(f"{c:.2f}: {100 * (peaks >= c).mean():5.1f}%" for c in cutoffs)
        label = "wakes on" if kind == "positive" else "FALSE wakes"
        print(f"{source:28s} ({len(peaks):3d} takes) {label:11s} {rates}")
if not glob.glob(f"{REAL_DIR}/*/*.wav"):
    print("No real recordings, so nothing to check. The training log above has the synthetic test results.")

## 7. Export for ESPHome

In [ ]:
# Writes hey_robin.tflite + hey_robin.json (v2 manifest) to Drive and downloads both.
# If ESPHome logs a tensor-arena error at boot, raise tensor_arena_size.
out = f"{DRIVE_DIR}/outputs/{MODEL_VERSION}"
os.makedirs(out, exist_ok=True)
shutil.copy(TFLITE, f"{out}/hey_robin.tflite")
manifest = {
    "type": "micro",
    "wake_word": "Hey Robin",
    "author": "Robin Assistant",
    "model": "hey_robin.tflite",
    "trained_languages": ["en"],
    "version": 2,
    "micro": {
        "probability_cutoff": PROBABILITY_CUTOFF,
        "sliding_window_size": 5,
        "feature_step_size": 10,
        "tensor_arena_size": 30000,
        "minimum_esphome_version": "2024.7.0",
    },
}
with open(f"{out}/hey_robin.json", "w") as f:
    json.dump(manifest, f, indent=2)
shutil.copy("training_parameters.yaml", f"{out}/training_parameters.yaml")
print("Saved to", out)

from google.colab import files
files.download(f"{out}/hey_robin.tflite")
files.download(f"{out}/hey_robin.json")